In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('../../src').resolve()))

In [ ]:
import download_data_mem
import processing
import utils
import pandas as pd
import numpy as np
import sys
import plot_radiosonde_ddu_map
import plot_radiosonde_linear 
import plot_radiosonde_skewt
from datetime import date, datetime, timedelta
from pathlib import Path
import importlib
import matplotlib.pyplot as plt
importlib.reload(download_data_mem)
importlib.reload(processing)
importlib.reload(utils)
importlib.reload(plot_radiosonde_linear)
importlib.reload(plot_radiosonde_skewt)

In [ ]:
winter = download_data_mem.get_data(years=[2020,2021,2022,2023,2024,2025],months=[6,7,8],hours = [0]) #DJF / MAM / JJA / SON iwith 2 cons. year
summer = download_data_mem.get_data(years=[2020,2021,2022,2023,2024,2025],months=[12,1,2],hours = [0])
transition = download_data_mem.get_data(years=[2020,2021,2022,2023,2024,2025],months=[3,4,5,9,10,11],hours = [0])

#CLUSTERING ?  CA.ENvoyer a heather les etudes sur le sujet.

In [ ]:
print("Winter:",len(winter),"Summer:",len(summer),"Transition:",len(transition))

In [ ]:
winter_int,winter_stats = utils.clean_and_interpolate(winter)
summer_int,summer_stats = utils.clean_and_interpolate(summer) #Check combien de profile sont clean , combein de nan values, etc..
transition_int,transition_stats = utils.clean_and_interpolate(transition) # aussi si c'est des tas de nan ou parse..
print("Winter:",len(winter_int),"Summer:",len(summer_int),"Transition:",len(transition_int))
#print(winter_int[0]['data'])


In [ ]:
print("Winter stats:",winter_stats[0])
print("Summer stats:",summer_stats[0])
print("Transition stats:",transition_stats[0])

In [ ]:
lines_removed=[]
for i,s in enumerate(summer_stats):
    lines_removed.append((i,s['len_before']-s['len_after']))
plt.hist([x[1] for x in lines_removed],bins=56)
plt.show()

In [ ]:
importlib.reload(utils)
utils.plot_vertical_coverage(winter_int)
utils.plot_vertical_coverage(summer_int)
utils.plot_vertical_coverage(transition_int)

In [ ]:
winter_stats[1]

In [ ]:
#Histogram of number of lines removed because of pressure removed
# QUASIMENT TOUS LES POINTS QUI ETAIENT REJETES ETAIENT DES PTS AUX PRESSIONS EGALES. J'AI CHANGé LA CONDITION POUR ACCEPTER CES PTS, A VOIR SI CEST OK...
fig,axs = plt.subplots(3,1,figsize=(15,10))
axs[0].hist([s['non_monotonic_pressure']/s['len_before'] for s in winter_stats], bins=100, label='Winter')
axs[1].hist([s['non_monotonic_pressure']/s['len_before'] for s in summer_stats], bins=100,  label='Summer')
axs[2].hist([s['non_monotonic_pressure']/s['len_before'] for s in transition_stats], bins=100, label='Transition')
plt.xlabel('Pressure Removed')
plt.ylabel('Number of Profiles')
plt.title('Distribution of fraction of lines removed because of non monotonic pressure in Profiles')
plt.legend()
plt.show()

#check comment sont repartis ces lignes sparse/dense.


also check profiles where there is saturation (RH) ?

In [ ]:
#multiplot of all profiles
plot_radiosonde_linear.plot_radiosonde_linear_multi(winter_int,plot_std=False)  
plot_radiosonde_linear.plot_radiosonde_linear_multi(summer_int,plot_std=False)
plot_radiosonde_linear.plot_radiosonde_linear_multi(transition_int,plot_std=False)

#essayer 2d histogram pour voir si multimodal (graph prof)

In [ ]:
display(winter_int[0]['data'][:80])

In [ ]:
importlib.reload(plot_radiosonde_skewt)
plot_radiosonde_skewt.plot_radiosonde_skewt_multi(winter_int[1:])
plot_radiosonde_skewt.plot_radiosonde_skewt_multi(summer_int)
plot_radiosonde_skewt.plot_radiosonde_skewt_multi(transition_int)

In [ ]:
plot_radiosonde_skewt.plot_radiosonde_skewt(summer_int[0])

In [ ]:
import matplotlib.pyplot as plt
pbl_winter = utils.pbl_height_distribution(winter_int)
pbl_summer = utils.pbl_height_distribution(summer_int)
pbl_transition = utils.pbl_height_distribution(transition_int)

print(pd.Series(pbl_winter).describe(),"\n")
print(pd.Series(pbl_summer).describe())
print(pd.Series(pbl_transition).describe())

#plt.hist(pbl_winter, bins=20, alpha=0.5, label='Winter')
#plt.hist(pbl_summer, bins=20, alpha=0.5, label='Summer')
plt.boxplot([pbl_winter, pbl_summer, pbl_transition], labels=['Winter', 'Summer', 'Transition'])

#plt.xlabel('PBL Height (m)')
#plt.ylabel('Frequency')
plt.title('PBL Height Distribution')
plt.legend()
plt.show()

In [ ]:
winter_inv = utils.inversion_statistics(winter_int)
summer_inv = utils.inversion_statistics(summer_int)
transition_inv = utils.inversion_statistics(transition_int)

print("Winter Inversion Statistics:", winter_inv)
print("Summer Inversion Statistics:", summer_inv)
print("Transition Inversion Statistics:", transition_inv)